In [ ]:
# ============================================================
# Paper 1 — Spatial CV and Scaling
# Cell 1: Imports, paths, and load the unscaled analytical table
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

TABLE_DIR = PROJECT_ROOT / "03_outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "03_outputs" / "figures"
QC_DIR = PROJECT_ROOT / "03_outputs" / "qc"

CV_DIR = PROJECT_ROOT / "03_outputs" / "spatial_cv"
CV_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load the unscaled 14-channel analytical table
# ------------------------------------------------------------

data_file = (
    TABLE_DIR /
    "Paper1_14ch_valid_cells_UNSCALED.csv"
)

df = pd.read_csv(data_file)

print("Loaded file:")
print(data_file)

print("\nDataframe shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nCell 1 completed.")

In [ ]:
# ============================================================
# Cell 2: Construct 5 geographically contiguous spatial folds
# ============================================================

# ------------------------------------------------------------
# Define spatial and feature columns
# ------------------------------------------------------------

spatial_columns = [
    "Cell_ID",
    "Row",
    "Col",
    "X",
    "Y"
]

feature_columns = [
    "RTE_TMI",
    "CBG",
    "CBG_RES",
    "HGM",
    "K",
    "eTh",
    "eU",
    "CPD",
    "MAG_LD",
    "GRAV_LD",
    "DEM_LD",
    "ID",
    "DEM",
    "SLOPE"
]

print("Number of feature channels:", len(feature_columns))

# ------------------------------------------------------------
# Inspect geographical span
# ------------------------------------------------------------

x_min = df["X"].min()
x_max = df["X"].max()

y_min = df["Y"].min()
y_max = df["Y"].max()

x_span_km = (x_max - x_min) / 1000.0
y_span_km = (y_max - y_min) / 1000.0

print("\nSpatial extent of valid analytical cells")
print("-" * 60)
print(f"X range : {x_min:.2f} to {x_max:.2f} m")
print(f"Y range : {y_min:.2f} to {y_max:.2f} m")
print(f"E-W span: {x_span_km:.1f} km")
print(f"N-S span: {y_span_km:.1f} km")

# ------------------------------------------------------------
# Assign 5 contiguous folds using X-coordinate quantiles.
#
# This creates west-to-east geographical blocks while keeping
# the number of valid cells in each fold approximately balanced.
# ------------------------------------------------------------

df_cv = df.copy()

df_cv["Spatial_Fold"] = pd.qcut(
    df_cv["X"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# ------------------------------------------------------------
# Summarize fold sizes and spatial ranges
# ------------------------------------------------------------

fold_summary = (
    df_cv
    .groupby("Spatial_Fold")
    .agg(
        n_cells=("Cell_ID", "count"),
        X_min=("X", "min"),
        X_max=("X", "max"),
        Y_min=("Y", "min"),
        Y_max=("Y", "max")
    )
    .reset_index()
)

fold_summary["X_width_km"] = (
    fold_summary["X_max"] -
    fold_summary["X_min"]
) / 1000.0

fold_summary["Y_span_km"] = (
    fold_summary["Y_max"] -
    fold_summary["Y_min"]
) / 1000.0

print("\nSpatial fold summary:")
display(fold_summary.round(2))

# ------------------------------------------------------------
# Plot spatial folds
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 7))

scatter = ax.scatter(
    df_cv["X"],
    df_cv["Y"],
    c=df_cv["Spatial_Fold"],
    s=18
)

ax.set_aspect("equal")

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_title(
    "Paper 1 — Initial 5-Fold Spatial Blocking\n"
    "Contiguous West-to-East Geographic Folds"
)

cbar = fig.colorbar(
    scatter,
    ax=ax,
    ticks=[1, 2, 3, 4, 5]
)

cbar.set_label("Spatial Fold")

plt.tight_layout()

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

fold_fig_output = (
    FIGURE_DIR /
    "Paper1_5fold_initial_spatial_blocks.png"
)

plt.savefig(
    fold_fig_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSpatial-fold figure saved to:")
print(fold_fig_output)

print("\nCell 2 completed.")

In [ ]:
# ============================================================
# Cell 3: Create buffered spatial train/validation partitions
#         for the main 13 x 13 patch configuration
# ============================================================

from scipy.spatial import cKDTree

# ------------------------------------------------------------
# Patch / grid settings
# ------------------------------------------------------------

CELL_SIZE_KM = 4.0
PATCH_SIZE = 13

# Strict no-overlap distance between training and validation
# patch centers:
BUFFER_KM = (PATCH_SIZE - 1) * CELL_SIZE_KM
BUFFER_M = BUFFER_KM * 1000.0

print("Patch size        :", PATCH_SIZE, "x", PATCH_SIZE)
print("Grid spacing      :", CELL_SIZE_KM, "km")
print("Exclusion buffer  :", BUFFER_KM, "km")

# ------------------------------------------------------------
# Coordinates of all valid cells
# ------------------------------------------------------------

all_xy = df_cv[["X", "Y"]].to_numpy()

fold_partition_records = []
fold_count_records = []

# ------------------------------------------------------------
# Build one train/buffer/validation partition per fold
# ------------------------------------------------------------

for fold in range(1, 6):

    # Validation cells for current fold
    val_mask = (
        df_cv["Spatial_Fold"].to_numpy() == fold
    )

    val_xy = all_xy[val_mask]

    # --------------------------------------------------------
    # Distance from every valid cell to nearest validation cell
    # --------------------------------------------------------

    tree = cKDTree(val_xy)

    nearest_distance_m, _ = tree.query(
        all_xy,
        k=1
    )

    # --------------------------------------------------------
    # Define roles
    #
    # VALIDATION:
    #   cells belonging to the held-out fold
    #
    # BUFFER:
    #   non-validation cells lying within BUFFER_M of
    #   any validation cell
    #
    # TRAIN:
    #   everything remaining
    # --------------------------------------------------------

    buffer_mask = (
        (~val_mask)
        & (nearest_distance_m <= BUFFER_M)
    )

    train_mask = (
        (~val_mask)
        & (~buffer_mask)
    )

    role = np.full(
        len(df_cv),
        "TRAIN",
        dtype=object
    )

    role[buffer_mask] = "BUFFER"
    role[val_mask] = "VALIDATION"

    # --------------------------------------------------------
    # Save fold-specific partition table
    # --------------------------------------------------------

    fold_df = df_cv[
        ["Cell_ID", "Row", "Col", "X", "Y", "Spatial_Fold"]
    ].copy()

    fold_df["CV_Role"] = role
    fold_df["Distance_to_validation_km"] = (
        nearest_distance_m / 1000.0
    )

    fold_df["Validation_Fold"] = fold

    fold_partition_records.append(fold_df)

    # --------------------------------------------------------
    # Count cells
    # --------------------------------------------------------

    n_train = int(train_mask.sum())
    n_buffer = int(buffer_mask.sum())
    n_val = int(val_mask.sum())

    fold_count_records.append({
        "Validation_Fold": fold,
        "Train_cells": n_train,
        "Buffer_cells": n_buffer,
        "Validation_cells": n_val,
        "Total_cells": len(df_cv)
    })

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

buffer_summary = pd.DataFrame(
    fold_count_records
)

display(buffer_summary)

print("\nCell 3 completed.")

In [ ]:
# ============================================================
# Cell 4: Rigorous no-overlap spatial buffer using raster
#         row/column coordinates for 13 x 13 patches
# ============================================================

PATCH_SIZE = 13

# Half-width of patch in cells
PATCH_RADIUS = (PATCH_SIZE - 1) // 2

# For two patches of equal size, their centers must be
# separated by > 2*PATCH_RADIUS cells in at least one axis
NO_OVERLAP_CELLS = 2 * PATCH_RADIUS

CELL_SIZE_KM = 4.0
NO_OVERLAP_KM = NO_OVERLAP_CELLS * CELL_SIZE_KM

print("Patch size             :", PATCH_SIZE, "x", PATCH_SIZE)
print("Patch radius           :", PATCH_RADIUS, "cells")
print("No-overlap separation  :", NO_OVERLAP_CELLS, "cells")
print("Equivalent distance    :", NO_OVERLAP_KM, "km")

# ------------------------------------------------------------
# Grid coordinates
# ------------------------------------------------------------

all_rows = df_cv["Row"].to_numpy()
all_cols = df_cv["Col"].to_numpy()
all_folds = df_cv["Spatial_Fold"].to_numpy()

strict_partition_records = []
strict_count_records = []

# ------------------------------------------------------------
# Loop through held-out validation folds
# ------------------------------------------------------------

for fold in range(1, 6):

    val_mask = (all_folds == fold)

    val_rows = all_rows[val_mask]
    val_cols = all_cols[val_mask]

    # Start with no cells marked as buffer
    buffer_mask = np.zeros(len(df_cv), dtype=bool)

    # --------------------------------------------------------
    # A candidate training center must be excluded if there is
    # ANY validation center for which:
    #
    # |row_train - row_val| <= NO_OVERLAP_CELLS
    # AND
    # |col_train - col_val| <= NO_OVERLAP_CELLS
    #
    # This is the correct square-patch overlap criterion.
    # --------------------------------------------------------

    candidate_indices = np.where(~val_mask)[0]

    for idx in candidate_indices:

        row_diff = np.abs(
            val_rows - all_rows[idx]
        )

        col_diff = np.abs(
            val_cols - all_cols[idx]
        )

        overlaps_any_validation_patch = np.any(
            (row_diff <= NO_OVERLAP_CELLS) &
            (col_diff <= NO_OVERLAP_CELLS)
        )

        if overlaps_any_validation_patch:
            buffer_mask[idx] = True

    # Validation cells themselves should not be buffer
    buffer_mask[val_mask] = False

    train_mask = (
        (~val_mask) &
        (~buffer_mask)
    )

    # --------------------------------------------------------
    # Assign role
    # --------------------------------------------------------

    role = np.full(
        len(df_cv),
        "TRAIN",
        dtype=object
    )

    role[buffer_mask] = "BUFFER"
    role[val_mask] = "VALIDATION"

    # --------------------------------------------------------
    # Save fold table
    # --------------------------------------------------------

    fold_df = df_cv[
        [
            "Cell_ID",
            "Row",
            "Col",
            "X",
            "Y",
            "Spatial_Fold"
        ]
    ].copy()

    fold_df["CV_Role"] = role
    fold_df["Validation_Fold"] = fold

    strict_partition_records.append(
        fold_df
    )

    # --------------------------------------------------------
    # Count roles
    # --------------------------------------------------------

    strict_count_records.append({
        "Validation_Fold": fold,
        "Train_cells": int(train_mask.sum()),
        "Buffer_cells": int(buffer_mask.sum()),
        "Validation_cells": int(val_mask.sum()),
        "Total_cells": len(df_cv)
    })

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

strict_buffer_summary = pd.DataFrame(
    strict_count_records
)

print("\nStrict patch-overlap buffer summary:")

display(strict_buffer_summary)

print("\nCell 4 completed.")

In [ ]:
# ============================================================
# Cell 5: Visualize strict TRAIN / BUFFER / VALIDATION geometry
#         using Fold 3 as the held-out validation region
# ============================================================

FOLD_TO_PLOT = 3

# ------------------------------------------------------------
# Retrieve the partition table for Fold 3
# strict_partition_records[0] = Fold 1
# strict_partition_records[1] = Fold 2
# etc.
# ------------------------------------------------------------

plot_df = strict_partition_records[
    FOLD_TO_PLOT - 1
].copy()

print(
    plot_df["CV_Role"]
    .value_counts()
)

# ------------------------------------------------------------
# Plot each role separately
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 7)
)

role_markers = {
    "TRAIN": "o",
    "BUFFER": "s",
    "VALIDATION": "^"
}

for role in [
    "TRAIN",
    "BUFFER",
    "VALIDATION"
]:

    subset = plot_df[
        plot_df["CV_Role"] == role
    ]

    ax.scatter(
        subset["X"],
        subset["Y"],
        s=20,
        marker=role_markers[role],
        label=f"{role} (n={len(subset)})"
    )

ax.set_aspect("equal")

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_title(
    f"Paper 1 — Buffered Spatial Cross-Validation\n"
    f"Fold {FOLD_TO_PLOT} held out | "
    f"{PATCH_SIZE}×{PATCH_SIZE} patch | "
    f"{NO_OVERLAP_KM:.0f}-km no-overlap separation"
)

ax.legend(
    loc="upper right"
)

plt.tight_layout()

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

partition_fig_output = (
    FIGURE_DIR /
    f"Paper1_Fold{FOLD_TO_PLOT}_Train_Buffer_Validation.png"
)

plt.savefig(
    partition_fig_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:")
print(partition_fig_output)

print("\nCell 5 completed.")

In [ ]:
# ============================================================
# Cell 6: Fold-specific Robust Scaling
#         Fit scaler on TRAIN only
# ============================================================

from sklearn.preprocessing import RobustScaler

# Containers for fold-specific objects/results
fold_scalers = {}
scaled_fold_data = {}
scaling_parameter_records = []

# ------------------------------------------------------------
# Loop through the 5 spatial CV folds
# ------------------------------------------------------------

for fold in range(1, 6):

    print("=" * 70)
    print(f"PROCESSING FOLD {fold}")
    print("=" * 70)

    # --------------------------------------------------------
    # Retrieve authoritative strict partition for this fold
    # --------------------------------------------------------

    partition_df = strict_partition_records[fold - 1].copy()

    # Add the 14 feature values by Cell_ID
    fold_data = partition_df.merge(
        df_cv[
            ["Cell_ID"] + feature_columns
        ],
        on="Cell_ID",
        how="left"
    )

    # --------------------------------------------------------
    # Separate TRAIN, BUFFER, VALIDATION
    # --------------------------------------------------------

    train_df = fold_data[
        fold_data["CV_Role"] == "TRAIN"
    ].copy()

    buffer_df = fold_data[
        fold_data["CV_Role"] == "BUFFER"
    ].copy()

    val_df = fold_data[
        fold_data["CV_Role"] == "VALIDATION"
    ].copy()

    X_train = train_df[feature_columns].to_numpy()
    X_buffer = buffer_df[feature_columns].to_numpy()
    X_val = val_df[feature_columns].to_numpy()

    # --------------------------------------------------------
    # Fit RobustScaler ONLY on training data
    #
    # RobustScaler:
    # center = median
    # scale  = IQR = Q75 - Q25
    # --------------------------------------------------------

    scaler = RobustScaler(
        with_centering=True,
        with_scaling=True,
        quantile_range=(25.0, 75.0)
    )

    scaler.fit(X_train)

    # Save fitted scaler in memory
    fold_scalers[fold] = scaler

    # --------------------------------------------------------
    # Transform data using TRAIN-derived parameters
    # --------------------------------------------------------

    X_train_scaled = scaler.transform(X_train)
    X_buffer_scaled = scaler.transform(X_buffer)
    X_val_scaled = scaler.transform(X_val)

    # --------------------------------------------------------
    # Save scaled dataframes
    # --------------------------------------------------------

    train_scaled_df = train_df[
        [
            "Cell_ID",
            "Row",
            "Col",
            "X",
            "Y",
            "Spatial_Fold",
            "CV_Role"
        ]
    ].copy()

    buffer_scaled_df = buffer_df[
        [
            "Cell_ID",
            "Row",
            "Col",
            "X",
            "Y",
            "Spatial_Fold",
            "CV_Role"
        ]
    ].copy()

    val_scaled_df = val_df[
        [
            "Cell_ID",
            "Row",
            "Col",
            "X",
            "Y",
            "Spatial_Fold",
            "CV_Role"
        ]
    ].copy()

    train_scaled_df[feature_columns] = X_train_scaled
    buffer_scaled_df[feature_columns] = X_buffer_scaled
    val_scaled_df[feature_columns] = X_val_scaled

    scaled_fold_data[fold] = {
        "train": train_scaled_df,
        "buffer": buffer_scaled_df,
        "validation": val_scaled_df
    }

    # --------------------------------------------------------
    # Store fold-specific scaling parameters
    # --------------------------------------------------------

    for i, channel in enumerate(feature_columns):

        scaling_parameter_records.append({
            "Validation_Fold": fold,
            "Channel": channel,
            "Training_Median": scaler.center_[i],
            "Training_IQR": scaler.scale_[i]
        })

    # --------------------------------------------------------
    # Diagnostic checks
    # --------------------------------------------------------

    train_scaled_median = np.median(
        X_train_scaled,
        axis=0
    )

    train_scaled_q25 = np.percentile(
        X_train_scaled,
        25,
        axis=0
    )

    train_scaled_q75 = np.percentile(
        X_train_scaled,
        75,
        axis=0
    )

    train_scaled_iqr = (
        train_scaled_q75 -
        train_scaled_q25
    )

    print("Train cells      :", len(train_df))
    print("Buffer cells     :", len(buffer_df))
    print("Validation cells :", len(val_df))

    print(
        "Maximum absolute TRAIN median after scaling:",
        round(np.max(np.abs(train_scaled_median)), 8)
    )

    print(
        "Maximum deviation of TRAIN IQR from 1:",
        round(np.max(np.abs(train_scaled_iqr - 1.0)), 8)
    )

# ------------------------------------------------------------
# Scaling parameter table
# ------------------------------------------------------------

scaling_params_df = pd.DataFrame(
    scaling_parameter_records
)

print("\n" + "=" * 70)
print("FOLD-SPECIFIC SCALING PARAMETERS")
print("=" * 70)

display(
    scaling_params_df.head(20).round(4)
)

# ------------------------------------------------------------
# Save scaling parameters
# ------------------------------------------------------------

scaling_params_output = (
    CV_DIR /
    "Paper1_5fold_RobustScaler_parameters.csv"
)

scaling_params_df.to_csv(
    scaling_params_output,
    index=False
)

print("\nScaling parameters saved to:")
print(scaling_params_output)

print("\nCell 6 completed.")

In [ ]:
# ============================================================
# Cell 7: Verify leakage-safe validation scaling
# ============================================================

leakage_records = []

for fold in range(1, 6):

    train_scaled = scaled_fold_data[fold]["train"]
    val_scaled = scaled_fold_data[fold]["validation"]

    # --------------------------------------------------------
    # Calculate medians and IQRs after scaling
    # --------------------------------------------------------

    train_values = train_scaled[feature_columns].to_numpy()
    val_values = val_scaled[feature_columns].to_numpy()

    train_median = np.median(train_values, axis=0)
    val_median = np.median(val_values, axis=0)

    train_iqr = (
        np.percentile(train_values, 75, axis=0)
        - np.percentile(train_values, 25, axis=0)
    )

    val_iqr = (
        np.percentile(val_values, 75, axis=0)
        - np.percentile(val_values, 25, axis=0)
    )

    for i, channel in enumerate(feature_columns):

        leakage_records.append({
            "Validation_Fold": fold,
            "Channel": channel,
            "Train_scaled_median": train_median[i],
            "Train_scaled_IQR": train_iqr[i],
            "Validation_scaled_median": val_median[i],
            "Validation_scaled_IQR": val_iqr[i]
        })

leakage_audit_df = pd.DataFrame(leakage_records)

print("Leakage-audit preview:")
display(
    leakage_audit_df.head(20).round(4)
)

# ------------------------------------------------------------
# Fold-level summary
# ------------------------------------------------------------

fold_leakage_summary = (
    leakage_audit_df
    .groupby("Validation_Fold")
    .agg(
        max_abs_train_median=(
            "Train_scaled_median",
            lambda x: np.max(np.abs(x))
        ),
        max_train_IQR_deviation=(
            "Train_scaled_IQR",
            lambda x: np.max(np.abs(x - 1.0))
        ),
        mean_abs_validation_median=(
            "Validation_scaled_median",
            lambda x: np.mean(np.abs(x))
        ),
        mean_abs_validation_IQR_deviation=(
            "Validation_scaled_IQR",
            lambda x: np.mean(np.abs(x - 1.0))
        )
    )
    .reset_index()
)

print("\nFold-level leakage audit:")
display(
    fold_leakage_summary.round(4)
)

# ------------------------------------------------------------
# Save audit
# ------------------------------------------------------------

leakage_output = (
    CV_DIR /
    "Paper1_5fold_scaling_leakage_audit.csv"
)

leakage_audit_df.to_csv(
    leakage_output,
    index=False
)

print("\nLeakage audit saved to:")
print(leakage_output)

print("\nCell 7 completed.")

In [ ]:
# ============================================================
# Cell 8: Save fold-specific scaled datasets
# ============================================================

saved_records = []

for fold in range(1, 6):

    fold_dir = (
        CV_DIR /
        f"Fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Retrieve scaled partitions
    # --------------------------------------------------------

    train_scaled_df = scaled_fold_data[fold]["train"]
    buffer_scaled_df = scaled_fold_data[fold]["buffer"]
    val_scaled_df = scaled_fold_data[fold]["validation"]

    # --------------------------------------------------------
    # Output paths
    # --------------------------------------------------------

    train_output = (
        fold_dir /
        f"Fold{fold}_TRAIN_scaled.csv"
    )

    buffer_output = (
        fold_dir /
        f"Fold{fold}_BUFFER_scaled.csv"
    )

    val_output = (
        fold_dir /
        f"Fold{fold}_VALIDATION_scaled.csv"
    )

    # --------------------------------------------------------
    # Save CSVs
    # --------------------------------------------------------

    train_scaled_df.to_csv(
        train_output,
        index=False
    )

    buffer_scaled_df.to_csv(
        buffer_output,
        index=False
    )

    val_scaled_df.to_csv(
        val_output,
        index=False
    )

    saved_records.append({
        "Fold": fold,
        "TRAIN_file": str(train_output),
        "TRAIN_cells": len(train_scaled_df),
        "BUFFER_file": str(buffer_output),
        "BUFFER_cells": len(buffer_scaled_df),
        "VALIDATION_file": str(val_output),
        "VALIDATION_cells": len(val_scaled_df)
    })

# ------------------------------------------------------------
# Save partition manifest
# ------------------------------------------------------------

fold_manifest_df = pd.DataFrame(
    saved_records
)

manifest_output = (
    CV_DIR /
    "Paper1_5fold_scaled_dataset_manifest.csv"
)

fold_manifest_df.to_csv(
    manifest_output,
    index=False
)

print("Scaled fold datasets saved successfully.\n")

display(
    fold_manifest_df[
        [
            "Fold",
            "TRAIN_cells",
            "BUFFER_cells",
            "VALIDATION_cells"
        ]
    ]
)

print("\nManifest saved to:")
print(manifest_output)

# ------------------------------------------------------------
# Final integrity check
# ------------------------------------------------------------

expected_total = len(df_cv)

for _, row in fold_manifest_df.iterrows():

    total = (
        row["TRAIN_cells"]
        + row["BUFFER_cells"]
        + row["VALIDATION_cells"]
    )

    assert total == expected_total, (
        f"Fold {row['Fold']} total mismatch"
    )

print(
    "\nIntegrity check passed: "
    "TRAIN + BUFFER + VALIDATION = 2718 "
    "for every fold."
)

print("\nCell 8 completed.")